In [3]:
import pandas as pd
data = pd.read_csv(r'C:\Users\pc\Downloads\mail_data.csv')
print(data.head())

  Category                                            Message
0      ham  Go until jurong point, crazy.. Available only ...
1      ham                      Ok lar... Joking wif u oni...
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...
3      ham  U dun say so early hor... U c already then say...
4      ham  Nah I don't think he goes to usf, he lives aro...


In [4]:
data.columns

Index(['Category', 'Message'], dtype='object')

In [5]:
data.isna().sum()

Category    0
Message     0
dtype: int64

In [6]:
data.duplicated().sum()

np.int64(415)

In [8]:
data.drop_duplicates()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [10]:
print(data.duplicated().sum())

415


In [11]:
data.drop_duplicates(inplace=True)

In [13]:
data.duplicated().sum()

np.int64(0)

In [21]:
print(data.columns)

Index(['Category', 'Message', 'cleaned'], dtype='object')


In [22]:
data = data.rename(columns={
    'Category': 'label',
    'Message': 'message'
})

In [24]:
data['label'] = data['label'].map({'ham': 0, 'spam':1})

In [25]:
import re
import nltk
from nltk.corpus import stopwords

In [26]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\pc\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [27]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9]', ' ', text)
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return ''.join(words)
data['cleaned'] = data['message'].apply(clean_text)

In [35]:
X = data['cleaned']
y = data['label']

In [37]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,y,
    test_size=0.2,
    random_state=42
)

In [38]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [39]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train_vec, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [40]:
from sklearn.metrics import classification_report, accuracy_score

y_pred = model.predict(X_test_vec)

print('Accuracy:', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.8682170542635659
              precision    recall  f1-score   support

           0       0.87      1.00      0.93       896
           1       0.00      0.00      0.00       136

    accuracy                           0.87      1032
   macro avg       0.43      0.50      0.46      1032
weighted avg       0.75      0.87      0.81      1032



C:\anacoda\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\anacoda\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\anacoda\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [41]:
from sklearn.pipeline import Pipeline 
import joblib

pipeline = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('model', MultinomialNB())
])
pipeline.fit(X_train, y_train)
joblib.dump(pipeline, 'spam_classifier.pkl')

['spam_classifier.pkl']

In [ ]:
loaded_model = joblib.load('spam_classifier.pkl')

msg ['Congratulations! You won a free iphone']
prediction = loaded_model.predict(msg)
print('Spam' if prediction[0]